# 02 - Patch extraction and tissue masking

**Owner:** Person 1


Illustrates the Otsu tissue detection and the 256x256 non-overlapping grid from
`src/preprocessing.py`, then shows the feature-extraction call that turns tiles
into the per-slide `.pt` bag every model consumes.


## Setup

In [ ]:
import os, sys, json
from pathlib import Path
os.environ.setdefault("MPLBACKEND", "Agg")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch
print("torch", torch.__version__)

## Otsu tissue detection on a thumbnail

In [ ]:
from PIL import Image
from src.preprocessing import detect_tissue_mask, otsu_threshold

thumb_dir = ROOT / "data/mock/thumbnails"
thumb_path = sorted(thumb_dir.glob("*.png"))[0] if thumb_dir.exists() else None
print("thumbnail:", thumb_path)
if thumb_path:
    thumb = Image.open(thumb_path).convert("RGB")
    mask = detect_tissue_mask(thumb)
    print("Otsu threshold:", round(otsu_threshold(np.asarray(thumb.convert("L"))), 1),
          "| tissue fraction:", round(float(mask.mean()), 3))
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    axes[0].imshow(thumb); axes[0].set_title("thumbnail"); axes[0].axis("off")
    axes[1].imshow(mask, cmap="gray"); axes[1].set_title("tissue mask"); axes[1].axis("off")
    plt.tight_layout()

## Tiling grid

In [ ]:
from src.preprocessing import grid_coordinates, select_tiles

dims = (4096, 4096)  # replace with real slide dimensions
tiles = grid_coordinates(*dims, patch_size=256)
print("grid tiles:", len(tiles))
if thumb_path:
    kept = select_tiles(mask, dims, patch_size=256, min_tissue_fraction=0.5)
    print("tiles passing the 50% tissue threshold:", len(kept))

## CLI equivalents (what actually runs in the pipeline)

In [ ]:
print('''# 1. tile one slide
python -m src.preprocessing --slide data/raw/<slide>.tiff --out data/patches/<slide_id>

# 2. extract frozen ResNet50 features for every tiled slide
python -m src.feature_extraction --patches-root data/patches --features-root data/features --backbone resnet50

# 3. build the frozen split (needs PANDA train.csv metadata)
python scripts/create_split.py --panda-csv data/raw/train.csv --out data/common_split.csv''')

## Rules that must not be broken

* Tiles are 256x256 with **no overlap**, matching the report's protocol.
* Only tissue tiles (>= 50% tissue) are kept; coordinates are saved so heatmaps work later.
* The encoder stays **frozen** and is applied identically to train/val/test slides.
* No pixel-level supervision is used anywhere in training.